# Word Count Pipeline — Sustainability Reports (Incremental)

Processes PDF annual reports from `data_ar_kam/`, extracts text (PyMuPDF + Gemini OCR for scanned pages), counts dictionary terms from `dt_kam_wordcount.csv`, and outputs structured word count results.

**Incremental mode:** Automatically detects unprocessed PDFs by comparing against the latest `results/` folder. Only new files are processed and results are merged with previous runs.

**OCR Modes:**
- `HYBRID` — PyMuPDF for text pages, Gemini OCR for image pages (default)
- `FULL_GEMINI_NOTES` — Gemini OCR for small docs (≤20 pages), PyMuPDF for large docs (with note)
- `FULL_GEMINI` — Gemini OCR for ALL pages

In [ ]:
!pip install PyMuPDF google-genai pandas tqdm Pillow

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

from src import config
from src.logger import setup_logger, get_logger
from src.utils import ensure_dirs, load_dictionary
from src.ocr_modes import OcrMode
from src.results_tracker import (
    get_latest_results_folder,
    load_processed_pairs,
    get_unprocessed_files,
)
from src.pipeline import run_pipeline

In [ ]:
# Configuration Preview
print("=" * 50)
print("Pipeline Configuration")
print("=" * 50)
print(f"PROJECT_ID:       {config.PROJECT_ID}")
print(f"MODEL_ID:         {config.MODEL_ID}")
print(f"LOCATION:         {config.LOCATION}")
print(f"BATCH_SIZE:       {config.BATCH_SIZE}")
print(f"MAX_WORKERS:      {config.MAX_WORKERS}")
print(f"OCR_IMAGE_DPI:    {config.OCR_IMAGE_DPI}")
print(f"MIN_TEXT_THRESH:   {config.MIN_TEXT_THRESHOLD}")
print(f"LARGE_DOC_THRESH: {config.LARGE_DOC_THRESHOLD}")
print(f"RESULTS_DIR:      {config.RESULTS_DIR}")
print()

# Verify paths
errors = []
if not config.PDF_DIR.exists():
    errors.append(f"PDF_DIR not found: {config.PDF_DIR}")
if not config.DICTIONARY_PATH.exists():
    errors.append(f"DICTIONARY_PATH not found: {config.DICTIONARY_PATH}")
if not config.SERVICE_ACCOUNT_PATH.exists():
    errors.append(f"SERVICE_ACCOUNT_PATH not found: {config.SERVICE_ACCOUNT_PATH}")

if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise FileNotFoundError("Missing required files. See errors above.")
else:
    print("All paths verified OK")
    print(f"  PDF_DIR:              {config.PDF_DIR}")
    print(f"  DICTIONARY_PATH:      {config.DICTIONARY_PATH}")
    print(f"  SERVICE_ACCOUNT_PATH: {config.SERVICE_ACCOUNT_PATH}")

In [ ]:
# Load Dictionary
dictionary_df = load_dictionary()
print(f"Shape: {dictionary_df.shape}")
print(f"Unique dimensions: {dictionary_df['Dimensions'].nunique()}")
print()
print("Sample rows:")
dictionary_df.head(10)

In [ ]:
# Scan for Unprocessed Files (Incremental Detection)
setup_logger()

# Find latest results folder (source of truth)
latest_folder = get_latest_results_folder(config.RESULTS_DIR)
print(f"Latest results folder: {latest_folder}")
print()

# Load already-processed (Emiten Code, Year) pairs
if latest_folder:
    processed_pairs = load_processed_pairs(latest_folder)
    print(f"Already processed: {len(processed_pairs)} (Emiten Code, Year) pairs")
else:
    processed_pairs = set()
    print("No previous results found — this will be the first run")

print()

# Find unprocessed PDFs
unprocessed = get_unprocessed_files(config.PDF_DIR, processed_pairs)
print(f"Unprocessed files: {len(unprocessed)}")
print()

if unprocessed:
    print("First 20 unprocessed files:")
    for f in unprocessed[:20]:
        print(f"  {f.name}")
    if len(unprocessed) > 20:
        print(f"  ... and {len(unprocessed) - 20} more")
else:
    print("All PDFs are already processed. Nothing to do.")

## Run Configuration

Choose the OCR mode and processing limits before running.

| Mode | Description | Use when |
|---|---|---|
| `HYBRID` | PyMuPDF text + Gemini OCR for image pages | Default, cost-efficient |
| `FULL_GEMINI_NOTES` | Gemini for small docs (≤20pp), PyMuPDF for large docs | Mixed dataset with large reports |
| `FULL_GEMINI` | Gemini OCR for ALL pages | Maximum accuracy, higher cost |

In [ ]:
# ============================================================
# RUN CONFIGURATION — Edit these values before running
# ============================================================

# OCR Mode: OcrMode.HYBRID | OcrMode.FULL_GEMINI_NOTES | OcrMode.FULL_GEMINI
ocr_mode = OcrMode.HYBRID

# Max files to process (None = all unprocessed)
max_files = None

# Files per batch
batch_size = 50

# Results folder label (None = auto from date)
results_label = None  # e.g. "march-2026-full-reports"

print(f"OCR Mode:     {ocr_mode.value}")
print(f"Max Files:    {max_files or 'all unprocessed'}")
print(f"Batch Size:   {batch_size}")
print(f"Label:        {results_label or 'auto'}")

## Run Pipeline

This cell runs the incremental pipeline. It:
1. Detects unprocessed PDFs (comparing against latest `results/` folder)
2. Processes only new files using the selected OCR mode
3. Merges results with previous run
4. Publishes to a new versioned `results/00x-*/` folder
5. Generates a diff report comparing with the previous run

In [ ]:
# Run the incremental pipeline
wordcount_df, summary_df = run_pipeline(
    max_files=max_files,
    batch_size=batch_size,
    ocr_mode=ocr_mode,
    results_label=results_label,
)

## Results

In [ ]:
# Word Count Results (cumulative)
print(f"Shape: {wordcount_df.shape}")
print(f"Unique Emiten Codes: {wordcount_df['Emiten Code'].nunique()}")
print(f"Unique Dimensions: {wordcount_df['Dimensions'].nunique()}")
print()

nonzero = wordcount_df[wordcount_df["Word count"] > 0]
print(f"Non-zero matches: {len(nonzero)} / {len(wordcount_df)} ({len(nonzero)/len(wordcount_df)*100:.1f}%)")
print(f"Total word count: {wordcount_df['Word count'].sum():,}")
print()

print("Top 20 matched terms:")
top_terms = nonzero.groupby("Wordlist")["Word count"].sum().nlargest(20)
display(top_terms)
print()

print("Head (20 rows):")
display(wordcount_df.head(20))

In [ ]:
# Process Summary
print(f"Shape: {summary_df.shape}")
print()

if not summary_df.empty:
    print("Status distribution:")
    print(summary_df["status"].value_counts())
    print()
    display(summary_df.head(10))

    # Show failed files if any
    failed = summary_df[summary_df["status"] == "failed"]
    if not failed.empty:
        print(f"\nFailed files ({len(failed)}):")
        for _, row in failed.iterrows():
            print(f"  {row['file_name']}: {row['error_message']}")
    else:
        print("\nNo failed files!")

In [ ]:
# View Diff Report
from src.results_tracker import get_latest_results_folder

new_results_folder = get_latest_results_folder(config.RESULTS_DIR)
print(f"Latest results folder: {new_results_folder}")

if new_results_folder:
    # Find the run report markdown
    md_files = list(new_results_folder.glob("*-run.md"))
    if md_files:
        from IPython.display import Markdown, display as ipy_display
        report_text = md_files[0].read_text(encoding="utf-8")
        ipy_display(Markdown(report_text))
    else:
        print("No run report found in the results folder.")
else:
    print("No results folder found.")

## Token Usage & Cost Analysis

In [ ]:
# Token Usage Analysis
token_path = config.TOKEN_USAGE_PATH
if token_path.exists():
    token_df = pd.read_csv(token_path)

    total_input = token_df["prompt_tokens"].sum()
    total_output = token_df["output_tokens"].sum()
    total_input_cost = total_input / 1_000_000 * config.PRICE_INPUT_PER_M
    total_output_cost = total_output / 1_000_000 * config.PRICE_OUTPUT_PER_M
    total_cost = total_input_cost + total_output_cost

    num_pdfs = len(summary_df[summary_df["status"] == "success"]) if not summary_df.empty else 1
    avg_cost_per_pdf = total_cost / num_pdfs if num_pdfs > 0 else 0

    print("=" * 50)
    print("Token Usage & Cost Summary (This Run)")
    print("=" * 50)
    print(f"Total OCR API calls:    {len(token_df)}")
    print(f"Total input tokens:     {total_input:,}")
    print(f"Total output tokens:    {total_output:,}")
    print(f"Input cost:             ${total_input_cost:.6f}")
    print(f"Output cost:            ${total_output_cost:.6f}")
    print(f"Total cost (this run):  ${total_cost:.6f}")
    print(f"Avg cost per PDF:       ${avg_cost_per_pdf:.6f}")
    print("=" * 50)
else:
    print("No token usage data found. Pipeline may not have used OCR.")

## Validation & Sanity Checks

In [ ]:
# Validation & Sanity Checks
warnings = []

# 1. PDF count
all_pdfs = list(config.PDF_DIR.glob("*.pdf"))
processed_count = len(summary_df) if not summary_df.empty else 0
print(f"1. PDFs found: {len(all_pdfs)}, In results: {processed_count}")

# 2. Dictionary entries
dict_count = len(dictionary_df)
print(f"2. Dictionary entries: {dict_count} (expected ~101)")

# 3. Duplicate check
if not wordcount_df.empty:
    dup_cols = ["Emiten Code", "Year", "Dimensions", "Wordlist"]
    dups = wordcount_df.duplicated(subset=dup_cols, keep=False)
    dup_count = dups.sum()
    print(f"3. Duplicate (code, year, dim, wordlist) rows: {dup_count}")
    if dup_count > 0:
        warnings.append(f"{dup_count} duplicate rows found")

# 4. Missing values
if not wordcount_df.empty:
    na_counts = wordcount_df[["Emiten Code", "Year", "Dimensions", "Wordlist", "Word count"]].isna().sum()
    total_na = na_counts.sum()
    print(f"4. Missing values in key columns: {total_na}")
    if total_na > 0:
        print(na_counts[na_counts > 0])
        warnings.append(f"{total_na} missing values found")

# 5. Zero-count analysis
if not wordcount_df.empty:
    term_totals = wordcount_df.groupby("Wordlist")["Word count"].sum()
    zero_terms = term_totals[term_totals == 0]
    print(f"5. Wordlist terms with 0 matches across ALL files: {len(zero_terms)}/{len(term_totals)}")
    if len(zero_terms) > 0:
        print(f"   Zero-match terms: {list(zero_terms.index[:20])}")

print()
if warnings:
    print(f"Warnings ({len(warnings)}):")
    for w in warnings:
        print(f"  - {w}")
else:
    print("All checks passed!")